# IndiGo DCF — InterGlobe Aviation (NSE: INDIGO)
## With War Scenario Stress Tests

**WACC:** 11% | **Terminal Growth Rate:** 4.5% | **Projection Period:** 5 years  
Data sourced live from Yahoo Finance via `yfinance`

---
## Step 1 — Pull Live Financial Data

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', palette='muted')

TICKER = 'INDIGO.NS'
ticker = yf.Ticker(TICKER)

# --- Raw financials ---
income_stmt = ticker.financials          # annual income statement
balance_sheet = ticker.balance_sheet     # annual balance sheet
cash_flow = ticker.cashflow              # annual cash flow statement
info = ticker.info

print(f"Fetched data for: {info.get('longName', TICKER)}")
print(f"Currency: {info.get('currency', 'INR')}")
print(f"Exchange: {info.get('exchange', 'NSI')}")

In [ ]:
# --------------------------------------------------------------------------
# Helper: safely extract a row from a DataFrame, return NaN series if missing
# --------------------------------------------------------------------------
def safe_row(df, candidates):
    """Try each label in candidates; return the first match or NaN series."""
    for label in candidates:
        if label in df.index:
            return df.loc[label]
    return pd.Series(np.nan, index=df.columns if df is not None and not df.empty else [])


# --------------------------------------------------------------------------
# Extract key line items (last 5 fiscal years, newest → oldest)
# --------------------------------------------------------------------------
cols = income_stmt.columns[:5]  # up to 5 most-recent years

revenue      = safe_row(income_stmt,   ['Total Revenue', 'Revenue'])[cols]
ebit         = safe_row(income_stmt,   ['EBIT', 'Operating Income', 'Operating Income Loss'])[cols]
da           = safe_row(cash_flow,     ['Depreciation And Amortization',
                                        'Depreciation Amortization Depletion',
                                        'Reconciled Depreciation'])[cols]
ebitda       = ebit + da

capex        = safe_row(cash_flow,     ['Capital Expenditure',
                                        'Purchase Of PPE',
                                        'Capital Expenditures'])[cols].abs()

# Working capital change (use operating cash flow proxy if unavailable)
wc_change    = safe_row(cash_flow,     ['Changes In Working Capital',
                                        'Change In Working Capital'])[cols]

tax_rate     = 0.25   # effective corporate tax rate (India)
nopat        = ebit * (1 - tax_rate)
fcf          = nopat + da - capex - wc_change

# --------------------------------------------------------------------------
# Market data
# --------------------------------------------------------------------------
current_price      = info.get('currentPrice') or info.get('regularMarketPrice', np.nan)
shares_outstanding = info.get('sharesOutstanding', np.nan)
market_cap         = info.get('marketCap', np.nan)

total_debt         = safe_row(balance_sheet, ['Total Debt',
                                               'Long Term Debt',
                                               'Total Long Term Debt']).iloc[0]
cash               = safe_row(balance_sheet, ['Cash And Cash Equivalents',
                                               'Cash Cash Equivalents And Short Term Investments',
                                               'Cash And Short Term Investments']).iloc[0]
net_debt           = total_debt - cash

# --------------------------------------------------------------------------
# Summary table  (values in INR Crores)
# --------------------------------------------------------------------------
CR = 1e7   # 1 Crore = 10 million

summary = pd.DataFrame({
    'Revenue (₹ Cr)':      (revenue / CR).round(0),
    'EBITDA (₹ Cr)':       (ebitda  / CR).round(0),
    'EBIT (₹ Cr)':         (ebit    / CR).round(0),
    'D&A (₹ Cr)':          (da      / CR).round(0),
    'CapEx (₹ Cr)':        (capex   / CR).round(0),
    'ΔWorking Capital (₹ Cr)': (wc_change / CR).round(0),
    'NOPAT (₹ Cr)':        (nopat   / CR).round(0),
    'FCF (₹ Cr)':          (fcf     / CR).round(0),
}).T

summary.columns = [str(c.year) for c in summary.columns]

print("="*65)
print(f"  InterGlobe Aviation (INDIGO.NS) — Historical Financials")
print("="*65)
print(summary.to_string())
print()
print(f"  Current Price      : ₹{current_price:,.2f}")
print(f"  Shares Outstanding : {shares_outstanding/1e7:,.2f} Cr shares")
print(f"  Market Cap         : ₹{market_cap/CR:,.0f} Cr")
print(f"  Total Debt         : ₹{total_debt/CR:,.0f} Cr")
print(f"  Cash               : ₹{cash/CR:,.0f} Cr")
print(f"  Net Debt           : ₹{net_debt/CR:,.0f} Cr")
print("="*65)

---
## Step 2 — Base Case DCF Model

In [ ]:
# --------------------------------------------------------------------------
# DCF assumptions — Base Case
# --------------------------------------------------------------------------
WACC         = 0.11   # 11% — INR-denominated, Indian aviation risk
TGR          = 0.045  # 4.5% terminal growth rate
PROJ_YEARS   = 5

# Use most-recent year as base
base_revenue = float(revenue.iloc[0])
base_ebitda  = float(ebitda.iloc[0])
base_ebitda_margin = base_ebitda / base_revenue if base_revenue else 0.18

# Analyst-style growth & margin assumptions (Base Case)
revenue_growth   = [0.12, 0.11, 0.10, 0.09, 0.08]   # tapering growth
ebitda_margins   = [base_ebitda_margin + 0.01 * i for i in range(PROJ_YEARS)]  # gradual expansion
da_pct_rev       = float(da.iloc[0]) / base_revenue if base_revenue else 0.10
capex_pct_rev    = float(capex.iloc[0]) / base_revenue if base_revenue else 0.08
wc_pct_rev       = 0.01   # modest working capital build

# --------------------------------------------------------------------------
# Project FCFs
# --------------------------------------------------------------------------
years       = list(range(1, PROJ_YEARS + 1))
proj_rev    = []
proj_ebitda = []
proj_ebit   = []
proj_fcf    = []

rev = base_revenue
for i in range(PROJ_YEARS):
    rev    = rev * (1 + revenue_growth[i])
    _ebitda = rev * ebitda_margins[i]
    _da     = rev * da_pct_rev
    _ebit   = _ebitda - _da
    _nopat  = _ebit * (1 - tax_rate)
    _capex  = rev * capex_pct_rev
    _wc     = rev * wc_pct_rev
    _fcf    = _nopat + _da - _capex - _wc
    proj_rev.append(rev)
    proj_ebitda.append(_ebitda)
    proj_ebit.append(_ebit)
    proj_fcf.append(_fcf)

# --------------------------------------------------------------------------
# Terminal value & enterprise value
# --------------------------------------------------------------------------
terminal_fcf = proj_fcf[-1] * (1 + TGR)
terminal_val = terminal_fcf / (WACC - TGR)

pv_fcf = sum(fcf_t / (1 + WACC)**t for t, fcf_t in zip(years, proj_fcf))
pv_tv  = terminal_val / (1 + WACC)**PROJ_YEARS
ev     = pv_fcf + pv_tv

equity_val      = ev - net_debt
implied_price   = equity_val / shares_outstanding if shares_outstanding else np.nan
upside_pct      = (implied_price / current_price - 1) * 100 if current_price else np.nan

# --------------------------------------------------------------------------
# Print DCF summary
# --------------------------------------------------------------------------
proj_df = pd.DataFrame({
    'Revenue (₹ Cr)':      [r/CR for r in proj_rev],
    'EBITDA (₹ Cr)':       [e/CR for e in proj_ebitda],
    'EBITDA Margin %':     [e/r*100 for e,r in zip(proj_ebitda, proj_rev)],
    'FCF (₹ Cr)':          [f/CR for f in proj_fcf],
}, index=[f'FY+{y}' for y in years])

print("="*65)
print("  Base Case — Projected Financials")
print("="*65)
print(proj_df.round(1).to_string())
print()
print(f"  PV of FCFs          : ₹{pv_fcf/CR:>10,.0f} Cr")
print(f"  PV of Terminal Value: ₹{pv_tv/CR:>10,.0f} Cr")
print(f"  Enterprise Value    : ₹{ev/CR:>10,.0f} Cr")
print(f"  Less: Net Debt      : ₹{net_debt/CR:>10,.0f} Cr")
print(f"  Equity Value        : ₹{equity_val/CR:>10,.0f} Cr")
print()
print(f"  Implied Share Price : ₹{implied_price:>10,.2f}")
print(f"  Current Price       : ₹{current_price:>10,.2f}")
print(f"  Upside / (Downside) : {upside_pct:>+10.1f}%")
print("="*65)

---
## Step 3 — War Scenario Stress Tests

In [ ]:
# --------------------------------------------------------------------------
# Scenario engine
# --------------------------------------------------------------------------
def run_scenario(name, revenue_growth_rates, ebitda_margin_adj,
                 capex_pct=None, wc_pct=0.01):
    """
    Run a DCF scenario.
    - revenue_growth_rates : list of 5 annual growth rates
    - ebitda_margin_adj    : list of 5 EBITDA margins (absolute, not delta)
    - capex_pct            : capex as % of revenue (default: base case)
    - wc_pct               : Δworking capital as % of revenue
    Returns dict with key outputs.
    """
    _capex_pct = capex_pct if capex_pct is not None else capex_pct_rev
    rev = base_revenue
    fcfs = []
    ebitdas = []
    revs = []
    for i in range(PROJ_YEARS):
        rev     = rev * (1 + revenue_growth_rates[i])
        _ebitda = rev * ebitda_margin_adj[i]
        _da     = rev * da_pct_rev
        _ebit   = _ebitda - _da
        _nopat  = _ebit * (1 - tax_rate)
        _capex  = rev * _capex_pct
        _wc     = rev * wc_pct
        _fcf    = _nopat + _da - _capex - _wc
        fcfs.append(_fcf)
        ebitdas.append(_ebitda)
        revs.append(rev)

    tv     = fcfs[-1] * (1 + TGR) / (WACC - TGR)
    pv_f   = sum(f / (1 + WACC)**t for t, f in enumerate(fcfs, 1))
    pv_t   = tv / (1 + WACC)**PROJ_YEARS
    ev_s   = pv_f + pv_t
    eq_s   = ev_s - net_debt
    price_s = eq_s / shares_outstanding if shares_outstanding else np.nan
    downside = (price_s / implied_price - 1) * 100
    avg_ebitda_margin = np.mean([e/r for e,r in zip(ebitdas, revs)]) * 100

    return {
        'Scenario': name,
        'Avg EBITDA Margin %': round(avg_ebitda_margin, 1),
        'EV (₹ Cr)': round(ev_s / CR, 0),
        'Equity Value (₹ Cr)': round(eq_s / CR, 0),
        'Implied Price (₹)': round(price_s, 2),
        'vs Base Case %': round(downside, 1),
        'vs Current Price %': round((price_s/current_price - 1)*100, 1) if current_price else np.nan,
    }


# --------------------------------------------------------------------------
# Scenario 1 — Base Case
# Crude $85, load factor 87%, revenue growth 12% → tapering
# --------------------------------------------------------------------------
s1 = run_scenario(
    name='Base Case (Crude $85, LF 87%)',
    revenue_growth_rates=[0.12, 0.11, 0.10, 0.09, 0.08],
    ebitda_margin_adj=[base_ebitda_margin + 0.01*i for i in range(5)],
)

# --------------------------------------------------------------------------
# Scenario 2 — Oil Shock
# Gulf conflict: crude spikes to $120, fuel costs +35%, yields compress 8%,
# load factor drops to 84%
# Fuel is ~35–40% of IndiGo cost base; +35% fuel = ~13-15pp EBITDA margin hit
# Yield compression reduces revenue ~8%
# --------------------------------------------------------------------------
oil_shock_margin = [max(base_ebitda_margin - 0.14 + 0.01*i, 0.03) for i in range(5)]  # recovers
oil_shock_growth = [-0.02, 0.04, 0.08, 0.10, 0.10]   # yr1 slight decline due to yield compression

s2 = run_scenario(
    name='Oil Shock (Crude $120, LF 84%)',
    revenue_growth_rates=oil_shock_growth,
    ebitda_margin_adj=oil_shock_margin,
)

# --------------------------------------------------------------------------
# Scenario 3 — Demand Collapse
# Regional conflict disrupts travel: LF drops to 72%, revenue -22%, 
# 3-year recovery
# --------------------------------------------------------------------------
demand_margin = [
    max(base_ebitda_margin - 0.20, 0.01),   # yr1: sharp hit (fixed cost leverage)
    max(base_ebitda_margin - 0.15, 0.03),   # yr2: still depressed
    base_ebitda_margin - 0.08,              # yr3: recovery begins
    base_ebitda_margin - 0.03,              # yr4: near-normal
    base_ebitda_margin,                     # yr5: recovered
]
demand_growth = [-0.22, 0.05, 0.12, 0.13, 0.11]   # deep drop then recovery

s3 = run_scenario(
    name='Demand Collapse (LF 72%, Rev -22%)',
    revenue_growth_rates=demand_growth,
    ebitda_margin_adj=demand_margin,
)

# --------------------------------------------------------------------------
# Results table
# --------------------------------------------------------------------------
results = pd.DataFrame([s1, s2, s3]).set_index('Scenario')
print("="*80)
print("  Scenario Analysis — IndiGo DCF Stress Test")
print("="*80)
print(results.to_string())
print("="*80)

---
## Step 4 — Visualisations

In [ ]:
# --------------------------------------------------------------------------
# Chart 1 — Scenario Implied Prices vs Current Price (tornado)
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('IndiGo DCF — War Scenario Stress Test', fontsize=14, fontweight='bold')

scenarios   = [s1, s2, s3]
names       = [s['Scenario'] for s in scenarios]
prices      = [s['Implied Price (₹)'] for s in scenarios]
vs_base_pct = [s['vs Base Case %'] for s in scenarios]
colors      = ['#2ecc71', '#e67e22', '#e74c3c']

# --- Left: Implied prices bar chart ---
ax1 = axes[0]
bars = ax1.barh(names, prices, color=colors, edgecolor='white', height=0.5)
ax1.axvline(current_price, color='navy', linestyle='--', linewidth=1.5, label=f'Current: ₹{current_price:,.0f}')
ax1.set_xlabel('Implied Share Price (₹)', fontsize=11)
ax1.set_title('Implied Share Price by Scenario', fontsize=12)
ax1.legend(fontsize=10)
for bar, price in zip(bars, prices):
    ax1.text(bar.get_width() + max(prices)*0.01, bar.get_y() + bar.get_height()/2,
             f'₹{price:,.0f}', va='center', fontsize=10, fontweight='bold')
ax1.set_xlim(0, max(prices) * 1.2)
ax1.invert_yaxis()

# --- Right: % downside vs base case ---
ax2 = axes[1]
bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in vs_base_pct]
bars2 = ax2.barh(names, vs_base_pct, color=bar_colors, edgecolor='white', height=0.5)
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('% Change vs Base Case', fontsize=11)
ax2.set_title('Upside / Downside vs Base Case (%)', fontsize=12)
for bar, val in zip(bars2, vs_base_pct):
    offset = 0.5 if val >= 0 else -0.5
    ax2.text(val + offset, bar.get_y() + bar.get_height()/2,
             f'{val:+.1f}%', va='center', ha='left' if val >= 0 else 'right',
             fontsize=10, fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('scenario_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: scenario_comparison.png")

In [ ]:
# --------------------------------------------------------------------------
# Chart 2 — Projected FCF by scenario over 5 years
# --------------------------------------------------------------------------
def get_fcf_series(revenue_growth_rates, ebitda_margin_adj, capex_pct=None, wc_pct=0.01):
    _capex_pct = capex_pct if capex_pct is not None else capex_pct_rev
    rev = base_revenue
    fcfs = []
    for i in range(PROJ_YEARS):
        rev     = rev * (1 + revenue_growth_rates[i])
        _ebitda = rev * ebitda_margin_adj[i]
        _da     = rev * da_pct_rev
        _ebit   = _ebitda - _da
        _nopat  = _ebit * (1 - tax_rate)
        _capex  = rev * _capex_pct
        _wc     = rev * wc_pct
        fcfs.append((_nopat + _da - _capex - _wc) / CR)
    return fcfs

fcf_base   = get_fcf_series([0.12,0.11,0.10,0.09,0.08],
                             [base_ebitda_margin + 0.01*i for i in range(5)])
fcf_oil    = get_fcf_series(oil_shock_growth, oil_shock_margin)
fcf_demand = get_fcf_series(demand_growth, demand_margin)

fig, ax = plt.subplots(figsize=(10, 5))
yr_labels = [f'FY+{i}' for i in range(1, 6)]

ax.plot(yr_labels, fcf_base,   'o-', color='#2ecc71', linewidth=2.5, markersize=8, label='Base Case')
ax.plot(yr_labels, fcf_oil,    's--', color='#e67e22', linewidth=2.5, markersize=8, label='Oil Shock')
ax.plot(yr_labels, fcf_demand, '^:', color='#e74c3c', linewidth=2.5, markersize=8, label='Demand Collapse')

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_title('Projected Free Cash Flow by Scenario (₹ Cr)', fontsize=13, fontweight='bold')
ax.set_ylabel('FCF (₹ Crores)', fontsize=11)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f} Cr'))
plt.tight_layout()
plt.savefig('fcf_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fcf_scenarios.png")

In [ ]:
# --------------------------------------------------------------------------
# Chart 3 — Waterfall: Value bridge from Base Case to each stress scenario
# --------------------------------------------------------------------------
base_price = s1['Implied Price (₹)']
oil_delta  = s2['Implied Price (₹)'] - base_price
dem_delta  = s3['Implied Price (₹)'] - base_price

labels  = ['Base Case', 'Oil Shock Impact', 'Oil Shock Price',
           'Demand Collapse Impact', 'Demand Collapse Price']
values  = [base_price, oil_delta, s2['Implied Price (₹)'],
           dem_delta,  s3['Implied Price (₹)']]
bottoms = [0,          base_price, 0,
           base_price, 0]
bar_c   = ['#2ecc71', '#e74c3c', '#e67e22',
           '#e74c3c', '#c0392b']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (lbl, val, bot, clr) in enumerate(zip(labels, values, bottoms, bar_c)):
    ax.bar(i, abs(val) if lbl.endswith('Impact') else val,
           bottom=bot if lbl.endswith('Impact') else 0,
           color=clr, edgecolor='white', width=0.55)
    y_pos = bot + val/2 if lbl.endswith('Impact') else val/2
    ax.text(i, max(values)*1.03 if not lbl.endswith('Impact') else bot + val + max(values)*0.01,
            f'₹{val:+,.0f}' if lbl.endswith('Impact') else f'₹{val:,.0f}',
            ha='center', fontsize=9, fontweight='bold')

ax.axhline(current_price, color='navy', linestyle='--', linewidth=1.5,
           label=f'Current Price ₹{current_price:,.0f}')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Share Price (₹)', fontsize=11)
ax.set_title('Value Bridge — Base Case vs War Stress Scenarios', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('value_bridge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: value_bridge.png")

---
## Summary

| Scenario | Implied Price | vs Base | vs Market |
|---|---|---|---|
| Base Case | See above | — | See above |
| Oil Shock ($120 crude) | See above | See above | See above |
| Demand Collapse (LF 72%) | See above | See above | See above |

**Key Takeaways:**
- IndiGo's high fuel cost exposure (~35-40% of opex) makes it acutely sensitive to oil price shocks.
- A demand collapse scenario (load factor 72%, revenue -22%) represents the most severe downside case given IndiGo's fixed cost structure and high operating leverage.
- The base case DCF assumes gradual margin improvement as fleet utilisation and ancillary revenue mature.
- All scenarios should be read as directional stress indicators, not investment advice.